# Notebook 01 — SD2.1 + cgDDI skin-lesion concepts + custom facial LoRA
**Kaggle T4 | SD2.1 FP16 | Compel 2.3.1 | 5 PNGs / prompt / mode.**

Four **independent condition-specific textual-inversion (TI) adapters** from cgDDI are supported: melanoma, basal-cell carcinoma, squamous-cell carcinoma, and actinic keratosis. The last is **premalignant**, not invasive skin cancer. None of these adapters has been validated here as a generator of full-face clinical photographs.

Compare **`base`**, **`ti`**, **`lora`**, **`combined`** with identical prompts, diffusion settings, and seeds. This notebook **does not** pretend that cgDDI TI adapters are trained LoRA weights. The optional LoRA comes from Notebook 02 and must use the same SD2.1 base. Long prompts are encoded through Compel with no silent truncation.

**Caution:** Output is synthetic research material, not diagnostic evidence. Skin cancer lesions are often visually indistinguishable from benign lesions, so expert review is needed before labeling any generated example as medically correct. Please check dataset usage terms and patient image consent.

## 1 · Install fixed dependency versions
In Kaggle, enable **T4 GPU** and **Internet ON**. Start a fresh kernel. Keep Kaggle's GPU-enabled PyTorch. If an import fails after installing, **restart the kernel once**, then rerun from the top.

In [ ]:
import subprocess, sys
REQUIREMENTS = [
    'diffusers==0.35.1', 'transformers==4.56.2', 'accelerate==1.10.1',
    'peft==0.17.1', 'huggingface_hub==0.34.4',
    'safetensors==0.6.2', 'ftfy==6.3.1', 'compel==2.3.1',
]
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *REQUIREMENTS])
print('Dependency setup finished. Use a fresh Kaggle kernel if imported modules remain stale.')

In [ ]:
import json, csv, re, time, gc, zipfile, hashlib
from pathlib import Path
from IPython.display import display
from PIL import Image
import torch, diffusers, transformers, accelerate, compel as compel_package
from diffusers import StableDiffusionPipeline, DPMSolverMultistepScheduler
from compel import Compel, ReturnedEmbeddingsType, DiffusersTextualInversionManager
from huggingface_hub import model_info
assert torch.cuda.is_available(), 'Enable GPU T4 in Kaggle Settings.'
print('GPU:', torch.cuda.get_device_name(0))
print('torch:', torch.__version__, 'diffusers:', diffusers.__version__,
      'transformers:', transformers.__version__, 'compel:', getattr(compel_package,'__version__','installed'))

## 2 · Config — change only the relevant fields
`MODES = ['base','ti']` works **before training**. After Notebook 02, add `lora` and `combined` and set your custom LoRA source. To experiment with one shared cancer LoRA, put the same directory under `LORA_SOURCES['shared']`. Adapters are **TI embeddings**, not pretrained LoRAs.

In [ ]:
BASE_MODEL = 'sd2-community/stable-diffusion-2-1-base'  # community SD2.1 mirror: document revision
PUBLIC_ADAPTERS = {
    'melanoma': 'hcarrion/melanoma',
    'bcc': 'hcarrion/basal_cell_carcinoma',
    'scc': 'hcarrion/squamous_cell_carcinoma',
    'ak': 'hcarrion/actinic_keratosis',  # premalignant, separate from cancer endpoint
}
# Leave blank until trained. Can point to a local .safetensors, folder, or HF repo.
LORA_SOURCES = {'melanoma': '', 'bcc': '', 'scc': '', 'ak': '', 'shared': ''}
MODES = ['base', 'ti']  # after training: ['base', 'ti', 'lora', 'combined']
LORA_SCALE = 0.8
OUT_DIR = Path('/kaggle/working/skin_cancer_generated')
IMAGES_PER_PROMPT = 5
WIDTH = HEIGHT = 512
STEPS = 30
GUIDANCE = 7.0
BASE_SEED = 60941
MAX_CONTENT_TOKENS = 150
MAX_CLIP_CHUNKS = 2
NEGATIVE_PROMPT = ('cartoon, drawing, illustration, artificial makeup, plastic skin, heavy airbrushing, '
                   'unrealistic facial anatomy, oversmoothed textures, blur, watermark, text')
assert 1 <= IMAGES_PER_PROMPT <= 25 and WIDTH % 8 == 0 and HEIGHT % 8 == 0
assert MODES and set(MODES).issubset({'base','ti','lora','combined'})
assert 0 <= LORA_SCALE <= 2
assert MAX_CONTENT_TOKENS == 150 and MAX_CLIP_CHUNKS == 2
# Prevent stale files from earlier runs contaminating the exported ZIP.
if OUT_DIR.exists() and any(OUT_DIR.iterdir()):
    raise RuntimeError(f'{OUT_DIR} already contains files. Change OUT_DIR to a new run name.')
OUT_DIR.mkdir(parents=True,exist_ok=True)

## 3. Load protected authored morphology prompts
Reads `prompts/` JSON files (300 prompts: 210 train, 45 val, 45 test; 60 independent families). Defaults to 5 train prompts; `PROMPT_LIMIT=None` selects all training prompts. Preserves original negative prompts and emits CSV+JSON review records. All generated images remain unapproved pending human QA.


In [ ]:
# Use project-authored 300 morphology prompts and protected prompt-family split.
from pathlib import Path
import sys,subprocess
DERMSYNREAL_REPO=Path('/kaggle/working/DermSynReal')
if not (DERMSYNREAL_REPO/'src/dermsynreal/kaggle_runs.py').is_file():
    subprocess.check_call(['git','clone','--depth','1','https://github.com/shanmukhdatta/DermSynReal.git',str(DERMSYNREAL_REPO)])
sys.path.insert(0,str(DERMSYNREAL_REPO/'src'))
from dermsynreal.kaggle_runs import prompt_collection,generation_manifest,append_run_log
PROMPT_PARTITION='train'
PROMPT_VARIANT='morphology'  # 'generic' is E1, 'morphology' is E2
PROMPT_OFFSET=0
PROMPT_LIMIT=5   # Kaggle pilot; None uses all 210 train prompts
PROMPTS=prompt_collection(DERMSYNREAL_REPO,'Skin_Cancer',partition=PROMPT_PARTITION,limit=PROMPT_LIMIT,offset=PROMPT_OFFSET,variant=PROMPT_VARIANT)
# Prevent E1/E2 or consecutive pilots from overwriting the same image set.
OUT_DIR=Path(OUT_DIR)/('prompt_'+PROMPT_VARIANT)
if OUT_DIR.exists():
    raise FileExistsError(f'Generation output already exists: {OUT_DIR}; choose a fresh run')
print('Authored prompts:',len(PROMPTS),'families:',len({p['prompt_family_id'] for p in PROMPTS}))


## 4 · Load base, condition-specific published TI embeddings and optional LoRAs
TI triggers are discovered from **actual tokenizer additions** rather than guessed. TI conditioning is activated **only** in `ti`/`combined` modes. The notebook verifies the exact base revision recorded with our custom LoRA, when provenance is available. For best reproducibility, pin repository revisions.

In [ ]:
base_revision = model_info(BASE_MODEL).sha
pipe = StableDiffusionPipeline.from_pretrained(
    BASE_MODEL, revision=base_revision, torch_dtype=torch.float16, use_safetensors=True
)
needed_targets = {p['subtype'] for p in PROMPTS} if set(MODES)&{'ti','combined'} else set()
adapter_tokens, adapter_revisions = {}, {}
for target in sorted(needed_targets):
    repo = PUBLIC_ADAPTERS[target]
    revision = model_info(repo).sha
    previous_vocab = set(pipe.tokenizer.get_vocab())
    pipe.load_textual_inversion(repo, revision=revision)
    new_tokens = sorted(set(pipe.tokenizer.get_vocab())-previous_vocab)
    if not new_tokens:
        raise RuntimeError(f'No textual-inversion token registered for {repo}')
    # Multi-vector TI can register token, token_1, token_2, ...
    primary = [s for s in new_tokens if not re.search(r'_\d+$', s)]
    if len(primary) != 1:
        raise RuntimeError(f'Ambiguous TI trigger tokens for {repo}: {new_tokens}. Inspect manually.')
    adapter_tokens[target] = primary[0]
    adapter_revisions[target] = revision
    print('TI loaded:', target, repo, 'trigger:', repr(primary[0]), 'token count:', len(new_tokens))

loaded_loras = {}
if set(MODES) & {'lora','combined'}:
    for target in sorted({p['subtype'] for p in PROMPTS}):
        source = str(LORA_SOURCES.get(target) or LORA_SOURCES.get('shared') or '').strip()
        if not source:
            raise FileNotFoundError(f'LoRA is required for subtype={target}; set LORA_SOURCES[{target!r}] or shared.')
        # If same shared LoRA is used for multiple subtypes, load it only once.
        if source in loaded_loras:
            continue
        path = Path(source)
        prov = (path.parent if path.is_file() else path) / 'training_provenance.json'
        if prov.is_file():
            p = json.loads(prov.read_text(encoding='utf-8'))
            if p.get('base_model') != BASE_MODEL or p.get('base_revision') != base_revision:
                raise ValueError('LoRA base model/revision does not match inference model.')
            if p.get('training_target') not in (target, 'all_labeled'):
                raise ValueError('Trained LoRA subtype does not match requested inference subtype.')
        else:
            print('WARNING: Missing provenance file for', source, '- compatibility is unverified.')
        name = 'lora_' + str(len(loaded_loras))
        if path.is_file():
            if path.suffix != '.safetensors':
                raise ValueError('Local LoRA file must be .safetensors')
            pipe.load_lora_weights(str(path.parent), weight_name=path.name, adapter_name=name)
        else:
            pipe.load_lora_weights(source, adapter_name=name)
        loaded_loras[source] = name
        print('Loaded LoRA:',name,'from',source)

pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)
pipe.enable_vae_slicing()
pipe.to('cuda')
pipe.text_encoder.eval()
print('Base revision:',base_revision)

## 5 · Strict Compel prompt handling (150 content tokens, 2 chunks)
The SD2.1 tokenizer has a 77-position CLIP window (75 content tokens plus BOS/EOS). Count tokens using the loaded tokenizer **after multi-vector TI expansion**. Reject >150 content tokens rather than truncating. Encode BOTH positive and negative prompts with Compel; equalize embedding lengths before passing into the pipeline.

In [ ]:
assert pipe.tokenizer.model_max_length == 77, 'Unexpected tokenizer CLIP length'
CONTENT_PER_CHUNK = 75
assert MAX_CONTENT_TOKENS == CONTENT_PER_CHUNK * MAX_CLIP_CHUNKS

ti_manager = DiffusersTextualInversionManager(pipe)
compel_encoder = Compel(
    tokenizer=pipe.tokenizer,
    text_encoder=pipe.text_encoder,
    textual_inversion_manager=ti_manager,
    returned_embeddings_type=ReturnedEmbeddingsType.LAST_HIDDEN_STATES_NORMALIZED,
    truncate_long_prompts=False,
    requires_pooled=False,
    device='cuda',
)

def build_positive(entry, mode):
    content = entry['text'].strip()
    if mode in {'ti','combined'}:
        token = adapter_tokens.get(entry['subtype'])
        if not token:
            raise RuntimeError(f'No TI token for {entry["subtype"]}')
        if token not in content:
            content += ', ' + token
    return content

def content_tokens(prompt):
    expanded = pipe.maybe_convert_prompt(prompt,pipe.tokenizer)
    return len(pipe.tokenizer(expanded, truncation=False, add_special_tokens=False)['input_ids'])

def check_prompt(prompt,label):
    if not isinstance(prompt,str) or not prompt.strip():
        raise ValueError(f'{label}: empty prompt')
    count = content_tokens(prompt)
    if count > MAX_CONTENT_TOKENS:
        raise ValueError(f'{label}: {count} content tokens; maximum {MAX_CONTENT_TOKENS}; shorten prompt.')
    return count

validated = {}
for entry in PROMPTS:
    for mode in MODES:
        p = build_positive(entry,mode)
        validated[(entry['id'],mode)] = (p, check_prompt(p,f'{entry["id"]}:{mode}'))
neg_count=max(check_prompt(entry.get('negative_prompt',NEGATIVE_PROMPT),entry['id']+'/negative') for entry in PROMPTS)
print('Token checks passed for',len(validated),'conditioned prompts; negative tokens:',neg_count)

## 6 · Generate 5 images for each prompt and mode
All modes share the same seed per image index so comparisons are controlled. LoRA is explicitly **disabled** in `base`/`ti` conditions. Each saved row records subtype, adapter, LoRA, seed, expanded-token count, and pending morphology review.

In [ ]:
def safe_id(s):
    return re.sub(r'[^a-zA-Z0-9_-]+','_',str(s)).strip('_')[:70] or 'prompt'

def set_mode(target,mode):
    if mode in {'lora','combined'}:
        source = str(LORA_SOURCES.get(target) or LORA_SOURCES.get('shared') or '').strip()
        pipe.set_adapters([loaded_loras[source]], adapter_weights=[LORA_SCALE])
        pipe.enable_lora()
    else:
        if loaded_loras:
            pipe.disable_lora()

records=[]
start=time.time()
for i,entry in enumerate(PROMPTS):
    target = entry['subtype']
    for mode in MODES:
        set_mode(target, mode)
        pos,pos_count = validated[(entry['id'],mode)]
        with torch.no_grad():
            pos_emb = compel_encoder(pos)
            neg_emb = compel_encoder(entry.get('negative_prompt',NEGATIVE_PROMPT))
            pos_emb,neg_emb = compel_encoder.pad_conditioning_tensors_to_same_length([pos_emb,neg_emb])
        if pos_emb.ndim != 3 or pos_emb.shape != neg_emb.shape:
            raise RuntimeError('Positive and negative embeddings have incompatible shapes')
        if pos_emb.shape[1] > 77*MAX_CLIP_CHUNKS:
            raise RuntimeError(f'Compel exceeded 2 chunks: {pos_emb.shape}')
        pos_emb = pos_emb.to(device='cuda',dtype=pipe.unet.dtype)
        neg_emb = neg_emb.to(device='cuda',dtype=pipe.unet.dtype)
        folder=OUT_DIR/safe_id(target)/safe_id(entry['id'])/mode
        folder.mkdir(parents=True,exist_ok=True)
        gallery=[]
        for j in range(IMAGES_PER_PROMPT):
            seed=BASE_SEED+i*1000+j  # mode-independent comparison seed
            rng=torch.Generator(device='cuda').manual_seed(seed)
            with torch.inference_mode():
                img=pipe(prompt_embeds=pos_emb,negative_prompt_embeds=neg_emb,
                         num_inference_steps=STEPS,guidance_scale=GUIDANCE,
                         height=HEIGHT,width=WIDTH,generator=rng).images[0]
            # Fail visibly if a model/runtime produces a near-black placeholder image.
            rgb_check = img.convert('RGB')
            if max(channel_max for _, channel_max in rgb_check.getextrema()) <= 10:
                raise RuntimeError(f'Near-black generation for {entry["id"]}, mode={mode}, seed={seed}; image not accepted')
            fp=folder/f'img_{j:02d}_seed_{seed}.png'
            img.save(fp)
            gallery.append(img)
            source = (str(LORA_SOURCES.get(target) or LORA_SOURCES.get('shared') or '')
                      if mode in {'lora','combined'} else None)
            records.append({
                'file':fp.relative_to(OUT_DIR).as_posix(), 'prompt_id':entry['id'],
                'requested_subtype':target,'premalignant':target=='ak','mode':mode,
                'prompt':pos,'negative_prompt':entry.get('negative_prompt',NEGATIVE_PROMPT),
                'prompt_content_tokens':pos_count,'negative_content_tokens':content_tokens(entry.get('negative_prompt',NEGATIVE_PROMPT)),
                'conditioning_positions':int(pos_emb.shape[1]),
                'seed':seed,'base_model':BASE_MODEL,'base_revision':base_revision,
                'textual_inversion':PUBLIC_ADAPTERS[target] if mode in {'ti','combined'} else None,
                'ti_revision':adapter_revisions.get(target) if mode in {'ti','combined'} else None,
                'ti_token':adapter_tokens.get(target) if mode in {'ti','combined'} else None,
                'lora_source':source,'lora_scale':LORA_SCALE if source else None,
                'steps':STEPS,'guidance':GUIDANCE,'image_size':[WIDTH,HEIGHT],
                'morphology_review':'pending','near_copy_review':'pending',
            })
        contact=Image.new('RGB',(WIDTH*len(gallery),HEIGHT))
        for k,img in enumerate(gallery): contact.paste(img,(k*WIDTH,0))
        showw=min(1200,contact.width)
        print('Preview:',entry['id'],mode)
        display(contact.resize((showw,round(contact.height*showw/contact.width))))
        del pos_emb,neg_emb
        gc.collect()
with (OUT_DIR/'metadata.jsonl').open('w',encoding='utf-8') as f:
    for r in records: f.write(json.dumps(r,ensure_ascii=False)+'\n')
print('Saved',len(records),'images in',round(time.time()-start,1),'seconds')

In [ ]:
# Export reproducible per-image CSV, SHA hashes, review draft and run provenance.
summary=generation_manifest(records,OUT_DIR/'reports',root=OUT_DIR,class_name='Skin_Cancer',repo_root=DERMSYNREAL_REPO,prompt_partition=PROMPT_PARTITION,model_id=BASE_MODEL,prompt_type=PROMPT_VARIANT)
append_run_log(OUT_DIR,kind='generation',config={'class':'Skin_Cancer','prompt_partition':PROMPT_PARTITION,'prompt_type':PROMPT_VARIANT,'prompt_count':len(PROMPTS),'images_per_prompt':IMAGES_PER_PROMPT,'generated':summary['generated_images']},artifacts=[OUT_DIR/'reports/generation_manifest.csv',OUT_DIR/'metadata.jsonl'],repo_root=DERMSYNREAL_REPO)
print('CSV:',OUT_DIR/'reports/generation_manifest.csv','Review draft:',OUT_DIR/'reports/synthetic_pool_review_draft.csv')

## 7 · Export dataset ZIP and verify counts
Includes images and metadata. Inspect lesion morphology and run copy/identity detection against real *training* images before selecting synthetic examples for the classifier.

In [ ]:
ZIP_PATH=Path('/kaggle/working/skin_cancer_generated.zip')
with zipfile.ZipFile(ZIP_PATH,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=6) as zf:
    for fp in sorted(OUT_DIR.rglob('*')):
        if fp.is_file(): zf.write(fp,fp.relative_to(OUT_DIR).as_posix())
assert len(records)==len(PROMPTS)*len(MODES)*IMAGES_PER_PROMPT
with zipfile.ZipFile(ZIP_PATH) as zf:
    assert zf.testzip() is None
print('Export:',ZIP_PATH,'images:',len(records))

## Checkpoint references and limitations
- cgDDI: https://github.com/hectorcarrion/ControllableGenDDI
- Melanoma: https://huggingface.co/hcarrion/melanoma
- BCC: https://huggingface.co/hcarrion/basal_cell_carcinoma
- SCC: https://huggingface.co/hcarrion/squamous_cell_carcinoma
- AK (premalignant only): https://huggingface.co/hcarrion/actinic_keratosis
- SD2.1 community mirror: https://huggingface.co/sd2-community/stable-diffusion-2-1-base
- Compel 2.3.1: https://pypi.org/project/compel/2.3.1/
- cgDDI pretrained TI concepts primarily target dermatological lesion imagery; **clinical full-face validity is unverified**. These generated images are **not suitable for diagnosis**. Keep AK separate from the cancer outcome.